# Chapter 4.6: Ranking Optimization Tricks

## Learning Objectives

By the end of this notebook, you will be able to:

1. Perform **feature importance analysis** and selection for ranking models
2. Apply **gradient compression** techniques for distributed ranking training
3. Implement **mixed-precision training** for ranking models with embeddings
4. Design **mixed-dimension embeddings** that allocate dimensions based on feature frequency
5. Implement embedding table compression: **hashing**, **quotient-remainder**, and **DHE**
6. Benchmark and compare these optimization techniques
7. Combine multiple optimization tricks for maximum efficiency

## Prerequisites

- Understanding of CTR prediction models
- Familiarity with embedding tables and their memory requirements
- Basic knowledge of distributed training concepts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part4/chapter_4.6_optimization_tricks.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part4/chapter_4.6_optimization_tricks.ipynb)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple
import time

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cpu')

In [ ]:
def generate_data_with_frequency(num_samples=20000, seed=42):
    """Generate CTR data with power-law feature frequency distribution."""
    np.random.seed(seed)
    
    # Features with different cardinalities and frequency distributions
    feature_configs = [
        {'name': 'user_id', 'vocab': 10000, 'zipf_param': 1.5},
        {'name': 'item_id', 'vocab': 50000, 'zipf_param': 1.2},
        {'name': 'category', 'vocab': 100, 'zipf_param': 0.8},
        {'name': 'brand', 'vocab': 5000, 'zipf_param': 1.3},
        {'name': 'city', 'vocab': 500, 'zipf_param': 1.0},
        {'name': 'device', 'vocab': 20, 'zipf_param': 0.5},
        {'name': 'os', 'vocab': 10, 'zipf_param': 0.3},
        {'name': 'hour', 'vocab': 24, 'zipf_param': 0.2},
    ]
    
    sparse_features = []
    for config in feature_configs:
        # Zipf distribution for realistic frequency patterns
        weights = np.arange(1, config['vocab'] + 1, dtype=float) ** (-config['zipf_param'])
        weights /= weights.sum()
        values = np.random.choice(config['vocab'], size=num_samples, p=weights)
        sparse_features.append(values)
    
    sparse = np.column_stack(sparse_features)
    dense = np.random.randn(num_samples, 5).astype(np.float32)
    
    logits = 0.1 * dense.sum(axis=1)
    for i in range(3):
        logits += 0.05 * np.log1p(sparse[:, i]) / np.log1p(feature_configs[i]['vocab'])
    probs = 1 / (1 + np.exp(-logits))
    labels = (np.random.rand(num_samples) < probs).astype(np.float32)
    
    return {
        'sparse': torch.LongTensor(sparse),
        'dense': torch.FloatTensor(dense),
        'labels': torch.FloatTensor(labels),
        'feature_configs': feature_configs
    }


data = generate_data_with_frequency()
print(f"Samples: {len(data['labels'])}, CTR: {data['labels'].mean():.3f}")
print("\nFeature statistics:")
for i, config in enumerate(data['feature_configs']):
    unique = len(torch.unique(data['sparse'][:, i]))
    print(f"  {config['name']:12s}: vocab={config['vocab']:>6d}, unique_in_data={unique:>6d}")

## 1. Feature Importance Analysis

Before optimizing a ranking model, understanding which features matter most helps prioritize optimization efforts.

### Methods
1. **Permutation importance**: Shuffle one feature, measure performance drop
2. **Gradient-based importance**: Measure average gradient magnitude per feature
3. **Embedding norm**: Use the L2 norm of learned embeddings as a proxy

In [ ]:
class BaseRankingModel(nn.Module):
    """Base ranking model for optimization experiments."""
    def __init__(self, feature_configs, embed_dim=16, num_dense=5, hidden_dims=None):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(cfg['vocab'], embed_dim) for cfg in feature_configs
        ])
        input_dim = embed_dim * len(feature_configs) + num_dense
        if hidden_dims is None:
            hidden_dims = [256, 128, 64]
        
        layers = []
        prev = input_dim
        for dim in hidden_dims:
            layers.extend([nn.Linear(prev, dim), nn.ReLU(), nn.Dropout(0.1)])
            prev = dim
        self.mlp = nn.Sequential(*layers)
        self.output = nn.Linear(prev, 1)
    
    def forward(self, sparse, dense):
        embeds = [self.embeddings[i](sparse[:, i]) for i in range(sparse.size(1))]
        x = torch.cat(embeds + [dense], dim=-1)
        return self.output(self.mlp(x)).squeeze(-1)


def train_base_model(model, data, epochs=10, batch_size=512, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    n = len(data['labels'])
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            optimizer.zero_grad()
            loss = criterion(model(data['sparse'][idx], data['dense'][idx]), data['labels'][idx])
            loss.backward()
            optimizer.step()
    return model


def compute_feature_importance(model, data, method='permutation', num_eval=2000):
    """Compute feature importance using different methods."""
    model.eval()
    idx = torch.arange(min(num_eval, len(data['labels'])))
    sparse = data['sparse'][idx]
    dense = data['dense'][idx]
    labels = data['labels'][idx]
    
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        base_logits = model(sparse, dense)
        base_loss = criterion(base_logits, labels).item()
    
    importances = []
    num_features = sparse.size(1)
    
    if method == 'permutation':
        for f in range(num_features):
            perm_sparse = sparse.clone()
            perm_sparse[:, f] = perm_sparse[torch.randperm(len(idx)), f]
            with torch.no_grad():
                perm_logits = model(perm_sparse, dense)
                perm_loss = criterion(perm_logits, labels).item()
            importances.append(perm_loss - base_loss)
    
    elif method == 'embedding_norm':
        for f in range(num_features):
            unique_ids = torch.unique(sparse[:, f])
            emb_weights = model.embeddings[f].weight[unique_ids]
            importances.append(emb_weights.norm(dim=-1).mean().item())
    
    return importances


# Train base model and compute importances
torch.manual_seed(42)
base_model = BaseRankingModel(data['feature_configs'])
base_model = train_base_model(base_model, data)

perm_importance = compute_feature_importance(base_model, data, method='permutation')
norm_importance = compute_feature_importance(base_model, data, method='embedding_norm')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = [cfg['name'] for cfg in data['feature_configs']]

axes[0].barh(names, perm_importance, color='steelblue')
axes[0].set_xlabel('Loss Increase')
axes[0].set_title('Permutation Importance')
axes[0].grid(True, alpha=0.3, axis='x')

axes[1].barh(names, norm_importance, color='coral')
axes[1].set_xlabel('Avg Embedding Norm')
axes[1].set_title('Embedding Norm Importance')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 2. Embedding Table Compression

Embedding tables often dominate model size. For a feature with vocabulary $V$ and dimension $d$, the table requires $O(Vd)$ parameters.

### 2.1 Hash Embedding

Map multiple items to the same embedding via hash function:

$$\mathbf{e}(x) = \mathbf{E}[h(x)]$$

where $h: V \to [m]$ maps vocabulary to $m \ll V$ buckets.

### 2.2 Quotient-Remainder (QR) Trick

Decompose each ID into quotient and remainder, embed each separately:

$$\mathbf{e}(x) = \mathbf{E}_q[x \div m] \odot \mathbf{E}_r[x \mod m]$$

This reduces memory from $O(Vd)$ to $O((V/m + m)d)$.

### 2.3 Deep Hash Embedding (DHE)

Use a neural network to generate embeddings from ID features:

$$\mathbf{e}(x) = \text{MLP}(\text{encode}(x))$$

where encode transforms the ID into a dense vector via hash functions.

> **⚠️ Common Pitfall:** Hash collisions in hash embedding can significantly degrade performance for high-frequency items. Consider using multiple hash functions and combining their embeddings.

In [ ]:
class HashEmbedding(nn.Module):
    """Hash-based embedding with multiple hash functions."""
    def __init__(self, num_items: int, embed_dim: int, num_buckets: int,
                 num_hashes: int = 2):
        super().__init__()
        self.num_hashes = num_hashes
        self.num_buckets = num_buckets
        
        # Multiple hash tables
        self.tables = nn.ModuleList([
            nn.Embedding(num_buckets, embed_dim) for _ in range(num_hashes)
        ])
        
        # Hash parameters (random primes for hashing)
        self.register_buffer('hash_a', torch.tensor([7919, 104729][:num_hashes]))
        self.register_buffer('hash_b', torch.tensor([12553, 45979][:num_hashes]))
    
    def _hash(self, ids, hash_idx):
        return (ids * self.hash_a[hash_idx] + self.hash_b[hash_idx]) % self.num_buckets
    
    def forward(self, ids):
        result = self.tables[0](self._hash(ids, 0))
        for i in range(1, self.num_hashes):
            result = result + self.tables[i](self._hash(ids, i))
        return result / self.num_hashes


class QREmbedding(nn.Module):
    """Quotient-Remainder embedding (Facebook, 2019)."""
    def __init__(self, num_items: int, embed_dim: int, num_buckets: int):
        super().__init__()
        self.num_buckets = num_buckets
        num_quotients = (num_items + num_buckets - 1) // num_buckets
        
        self.quotient_table = nn.Embedding(num_quotients, embed_dim)
        self.remainder_table = nn.Embedding(num_buckets, embed_dim)
    
    def forward(self, ids):
        quotient = ids // self.num_buckets
        remainder = ids % self.num_buckets
        return self.quotient_table(quotient) * self.remainder_table(remainder)


class DHEmbedding(nn.Module):
    """Deep Hash Embedding."""
    def __init__(self, num_items: int, embed_dim: int, num_hash_features: int = 16):
        super().__init__()
        self.num_hash_features = num_hash_features
        
        # Random projection for encoding IDs
        self.register_buffer(
            'hash_proj',
            torch.randn(1, num_hash_features)
        )
        self.register_buffer(
            'hash_bias',
            torch.randn(num_hash_features) * np.pi
        )
        
        # MLP to generate embedding from hash features
        self.mlp = nn.Sequential(
            nn.Linear(num_hash_features, embed_dim * 2),
            nn.ReLU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )
    
    def encode_ids(self, ids):
        """Encode IDs into dense features using periodic functions."""
        ids_float = ids.float().unsqueeze(-1)  # (..., 1)
        features = torch.sin(ids_float * self.hash_proj + self.hash_bias)  # (..., num_hash)
        return features
    
    def forward(self, ids):
        features = self.encode_ids(ids)
        return self.mlp(features)


# Compare memory usage
vocab = 50000
dim = 16
buckets = 1000

methods = {
    'Standard': nn.Embedding(vocab, dim),
    'Hash (2 tables)': HashEmbedding(vocab, dim, buckets, num_hashes=2),
    'QR': QREmbedding(vocab, dim, buckets),
    'DHE': DHEmbedding(vocab, dim),
}

print(f"Embedding compression comparison (vocab={vocab:,}, dim={dim}):")
print(f"{'Method':<20} {'Parameters':>12} {'Compression':>12}")
print("-" * 46)
std_params = sum(p.numel() for p in methods['Standard'].parameters())
for name, module in methods.items():
    params = sum(p.numel() for p in module.parameters())
    ratio = std_params / params
    print(f"{name:<20} {params:>12,} {ratio:>11.1f}x")

## 3. Mixed-Dimension Embeddings

Not all features need the same embedding dimension. High-frequency features benefit from larger dimensions, while rare features can use smaller dimensions.

### Allocation Strategy

Given a total embedding parameter budget $B$, allocate dimension $d_f$ to feature $f$ proportional to:

$$d_f \propto \sqrt{\log(\text{freq}_f + 1)}$$

subject to $\sum_f V_f \cdot d_f \leq B$.

> **🔑 Pro Tip:** Mixed-dimension embeddings are especially effective when features have very different cardinalities (e.g., user_id: 1M vs hour_of_day: 24). The rare-feature dimensions can be much smaller without quality loss.

In [ ]:
class MixedDimensionEmbedding(nn.Module):
    """Embedding layer with feature-specific dimensions."""
    def __init__(self, feature_configs: List[Dict], output_dim: int,
                 total_budget: int = None, min_dim: int = 4, max_dim: int = 64):
        super().__init__()
        self.output_dim = output_dim
        
        if total_budget is None:
            total_budget = sum(cfg['vocab'] * 16 for cfg in feature_configs)
        
        # Compute frequency-based dimension allocation
        log_vocabs = [np.sqrt(np.log(cfg['vocab'] + 1)) for cfg in feature_configs]
        total_weight = sum(log_vocabs)
        
        # Allocate dimensions proportionally to importance (inverse of vocab for dense signal)
        dims = []
        for cfg, lv in zip(feature_configs, log_vocabs):
            # Higher dimension for smaller vocab (denser signal per parameter)
            raw_dim = total_budget * (lv / total_weight) / cfg['vocab']
            dim = int(np.clip(raw_dim, min_dim, max_dim))
            # Round to nearest multiple of 4
            dim = max(min_dim, (dim // 4) * 4)
            dims.append(dim)
        
        self.dims = dims
        self.embeddings = nn.ModuleList([
            nn.Embedding(cfg['vocab'], dim)
            for cfg, dim in zip(feature_configs, dims)
        ])
        
        # Projection to uniform output dimension
        self.projections = nn.ModuleList([
            nn.Linear(dim, output_dim) if dim != output_dim else nn.Identity()
            for dim in dims
        ])
    
    def forward(self, sparse_features: torch.Tensor) -> torch.Tensor:
        embeds = []
        for i in range(sparse_features.size(1)):
            emb = self.embeddings[i](sparse_features[:, i])
            emb = self.projections[i](emb)
            embeds.append(emb)
        return torch.cat(embeds, dim=-1)
    
    def get_stats(self, feature_configs):
        """Print dimension allocation statistics."""
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Total parameters: {total_params:,}")
        print(f"\n{'Feature':<12} {'Vocab':>8} {'Dim':>6} {'Params':>10}")
        print("-" * 40)
        for i, (cfg, dim) in enumerate(zip(feature_configs, self.dims)):
            emb_params = sum(p.numel() for p in self.embeddings[i].parameters())
            proj_params = sum(p.numel() for p in self.projections[i].parameters())
            print(f"{cfg['name']:<12} {cfg['vocab']:>8,} {dim:>6} {emb_params + proj_params:>10,}")


# Compare uniform vs mixed dimensions
feature_configs = data['feature_configs']
uniform_embed = BaseRankingModel(feature_configs, embed_dim=16)
uniform_params = sum(p.numel() for p in uniform_embed.parameters())

mixed_embed = MixedDimensionEmbedding(feature_configs, output_dim=16, total_budget=uniform_params // 2)
print("Mixed-Dimension Allocation:")
mixed_embed.get_stats(feature_configs)

print(f"\nUniform embedding parameters: {sum(sum(p.numel() for p in e.parameters()) for e in uniform_embed.embeddings):,}")
print(f"Mixed embedding parameters: {sum(p.numel() for p in mixed_embed.parameters()):,}")

## 4. Mixed-Precision Training

Mixed-precision training uses FP16 for most computations and FP32 for critical operations (loss computation, gradient accumulation), reducing memory and computation.

### Key Considerations for Ranking Models

1. **Embeddings**: Can be stored in FP16 (large memory savings)
2. **MLP layers**: FP16 forward/backward, FP32 master weights
3. **Loss computation**: Must be FP32 for numerical stability
4. **Gradient scaling**: Required to prevent FP16 gradient underflow

> **💡 Concept:** For ranking models with large embedding tables, FP16 embeddings alone can cut model memory by nearly 50%, as embeddings often comprise 80%+ of model parameters.

In [ ]:
class MixedPrecisionWrapper:
    """Simulate mixed-precision training behavior."""
    def __init__(self, model, loss_scale=1024.0):
        self.model = model
        self.loss_scale = loss_scale
        
        # Track precision of different components
        self.precision_map = {}
        for name, param in model.named_parameters():
            if 'embedding' in name:
                self.precision_map[name] = {'storage': 'FP16', 'compute': 'FP16'}
            elif 'output' in name or 'bias' in name:
                self.precision_map[name] = {'storage': 'FP32', 'compute': 'FP32'}
            else:
                self.precision_map[name] = {'storage': 'FP32', 'compute': 'FP16'}
    
    def compute_memory_savings(self):
        """Estimate memory savings from mixed precision."""
        fp32_total = 0
        mixed_total = 0
        
        for name, param in self.model.named_parameters():
            num_bytes_fp32 = param.numel() * 4
            fp32_total += num_bytes_fp32
            
            if self.precision_map.get(name, {}).get('storage') == 'FP16':
                mixed_total += param.numel() * 2
            else:
                mixed_total += num_bytes_fp32
        
        return fp32_total, mixed_total
    
    def print_precision_report(self):
        """Print precision assignment for each layer."""
        print(f"{'Layer':<40} {'Storage':>8} {'Compute':>8} {'Params':>10}")
        print("-" * 70)
        for name, param in self.model.named_parameters():
            info = self.precision_map.get(name, {'storage': 'FP32', 'compute': 'FP32'})
            print(f"{name:<40} {info['storage']:>8} {info['compute']:>8} {param.numel():>10,}")


mp_wrapper = MixedPrecisionWrapper(base_model)
fp32_mem, mixed_mem = mp_wrapper.compute_memory_savings()
print(f"FP32 memory: {fp32_mem / 1024 / 1024:.2f} MB")
print(f"Mixed memory: {mixed_mem / 1024 / 1024:.2f} MB")
print(f"Savings: {(1 - mixed_mem / fp32_mem) * 100:.1f}%")
print()
mp_wrapper.print_precision_report()

## 5. Gradient Compression for Distributed Training

In distributed ranking model training, communication of gradients is a major bottleneck, especially for embedding gradients which are sparse.

### Techniques

1. **Top-K sparsification**: Only communicate the K largest gradients
2. **Quantization**: Reduce gradient precision (e.g., 1-bit SGD)
3. **Error feedback**: Accumulate compression errors and add them back

In [ ]:
class GradientCompressor:
    """Gradient compression for distributed training."""
    def __init__(self, compression_ratio=0.1):
        self.compression_ratio = compression_ratio
        self.error_feedback = {}  # Accumulated compression errors
    
    def topk_compress(self, gradient: torch.Tensor, name: str) -> Tuple[torch.Tensor, float]:
        """Top-K gradient sparsification with error feedback."""
        # Add error feedback from previous step
        if name in self.error_feedback:
            gradient = gradient + self.error_feedback[name]
        
        # Select top-K values
        flat = gradient.abs().flatten()
        k = max(1, int(flat.numel() * self.compression_ratio))
        threshold = torch.topk(flat, k).values[-1]
        
        mask = gradient.abs() >= threshold
        compressed = gradient * mask.float()
        
        # Store error for feedback
        self.error_feedback[name] = gradient - compressed
        
        # Compute actual compression ratio
        nonzero = mask.sum().item()
        actual_ratio = nonzero / gradient.numel()
        
        return compressed, actual_ratio
    
    def quantize_1bit(self, gradient: torch.Tensor) -> Tuple[torch.Tensor, float]:
        """1-bit quantization: sign + mean magnitude."""
        mean_abs = gradient.abs().mean()
        sign = gradient.sign()
        quantized = sign * mean_abs
        # 1 bit per value + 1 float for mean = ~32x compression
        compression = 32.0  # FP32 -> 1 bit
        return quantized, compression


# Demonstrate gradient compression
compressor = GradientCompressor(compression_ratio=0.01)

# Simulate gradient computation
model = BaseRankingModel(data['feature_configs'])
logits = model(data['sparse'][:256], data['dense'][:256])
loss = F.binary_cross_entropy_with_logits(logits, data['labels'][:256])
loss.backward()

print("Gradient Compression Results:")
print(f"{'Layer':<30} {'Original':>10} {'Compressed':>10} {'Ratio':>8}")
print("-" * 62)
total_original = 0
total_compressed = 0

for name, param in model.named_parameters():
    if param.grad is not None:
        orig_bytes = param.grad.numel() * 4
        compressed, ratio = compressor.topk_compress(param.grad, name)
        comp_bytes = int(orig_bytes * ratio) + 4  # indices + values
        total_original += orig_bytes
        total_compressed += comp_bytes
        if 'embedding' in name or 'output' in name:
            print(f"{name:<30} {orig_bytes:>10,} {comp_bytes:>10,} {ratio:>7.3f}")

print(f"\nTotal: {total_original:,} -> {total_compressed:,} bytes ({total_compressed/total_original*100:.1f}%)")

## 6. Comprehensive Benchmark

In [ ]:
class CompressedRankingModel(nn.Module):
    """Ranking model with embedding compression."""
    def __init__(self, feature_configs, embed_dim=16, num_dense=5,
                 compression='none', num_buckets=1000):
        super().__init__()
        self.compression = compression
        
        if compression == 'none':
            self.embeddings = nn.ModuleList([
                nn.Embedding(cfg['vocab'], embed_dim) for cfg in feature_configs
            ])
        elif compression == 'hash':
            self.embeddings = nn.ModuleList([
                HashEmbedding(cfg['vocab'], embed_dim, min(num_buckets, cfg['vocab']))
                for cfg in feature_configs
            ])
        elif compression == 'qr':
            self.embeddings = nn.ModuleList([
                QREmbedding(cfg['vocab'], embed_dim, min(num_buckets, cfg['vocab']))
                for cfg in feature_configs
            ])
        elif compression == 'dhe':
            self.embeddings = nn.ModuleList([
                DHEmbedding(cfg['vocab'], embed_dim) for cfg in feature_configs
            ])
        
        input_dim = embed_dim * len(feature_configs) + num_dense
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, sparse, dense):
        embeds = [self.embeddings[i](sparse[:, i]) for i in range(sparse.size(1))]
        x = torch.cat(embeds + [dense], dim=-1)
        return self.mlp(x).squeeze(-1)


def benchmark_model(model, data, epochs=10, batch_size=512, lr=1e-3):
    """Train and evaluate, return AUC and timing."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    n = len(data['labels'])
    
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            optimizer.zero_grad()
            loss = criterion(model(data['sparse'][idx], data['dense'][idx]), data['labels'][idx])
            loss.backward()
            optimizer.step()
    train_time = time.time() - start_time
    
    model.eval()
    with torch.no_grad():
        preds = torch.sigmoid(model(data['sparse'][:2000], data['dense'][:2000])).numpy()
        labels = data['labels'][:2000].numpy()
        pos = preds[labels == 1]
        neg = preds[labels == 0]
        auc = np.mean(pos[:, None] > neg[None, :200]) if len(pos) > 0 and len(neg) > 0 else 0.5
    
    return auc, train_time


# Benchmark all compression methods
compressions = ['none', 'hash', 'qr', 'dhe']
benchmark_results = []

for comp in compressions:
    torch.manual_seed(42)
    model = CompressedRankingModel(data['feature_configs'], compression=comp)
    params = sum(p.numel() for p in model.parameters())
    auc, train_time = benchmark_model(model, data)
    benchmark_results.append({
        'method': comp, 'params': params, 'auc': auc, 'time': train_time
    })
    print(f"{comp:10s} | Params: {params:>10,} | AUC: {auc:.4f} | Time: {train_time:.2f}s")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

method_names = [r['method'] for r in benchmark_results]
params_list = [r['params'] for r in benchmark_results]
auc_list = [r['auc'] for r in benchmark_results]

colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
axes[0].bar(method_names, params_list, color=colors)
axes[0].set_ylabel('Parameters')
axes[0].set_title('Model Size by Compression Method')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(method_names, auc_list, color=colors)
axes[1].set_ylabel('Test AUC')
axes[1].set_title('Quality by Compression Method')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(min(auc_list) - 0.02, max(auc_list) + 0.02)

plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Mixed-Dimension Embeddings and Benchmark

Implement a ranking model with mixed-dimension embeddings and compare against uniform dimensions at the same parameter budget.

In [ ]:
def compare_mixed_vs_uniform(feature_configs, data, budgets=None):
    """Compare mixed-dimension vs uniform embeddings at various budgets.
    
    TODO:
    1. For each budget level, create:
       - Uniform embedding model (same dim for all features)
       - Mixed embedding model (frequency-based allocation)
    2. Train both and compare AUC
    3. Plot budget vs AUC for both approaches
    4. Print the dimension allocation for each budget level
    """
    if budgets is None:
        budgets = [100000, 200000, 500000, 1000000]
    
    # TODO: Implement comparison
    pass


# TODO: compare_mixed_vs_uniform(data['feature_configs'], data)

### 🏋️ Exercise 2: Implement Compositional Embedding

Implement a compositional embedding where each ID is represented as a weighted sum of base embeddings.

In [ ]:
class CompositionalEmbedding(nn.Module):
    """Compositional embedding: represent each ID as a combination of base vectors.
    
    Each ID x is decomposed into K components:
        e(x) = sum_k alpha_k(x) * B_k[h_k(x)]
    
    where B_k are base embedding tables, h_k are hash functions,
    and alpha_k are learned combination weights.
    """
    def __init__(self, num_items: int, embed_dim: int,
                 num_components: int = 4, base_size: int = 256):
        super().__init__()
        # TODO: Implement compositional embedding
        # 1. Create K base embedding tables of size base_size
        # 2. Create hash functions to map IDs to base indices
        # 3. Create a weight prediction network
        pass
    
    def forward(self, ids):
        # TODO: Implement forward pass
        # 1. Hash IDs into base indices for each component
        # 2. Look up base embeddings
        # 3. Compute combination weights
        # 4. Return weighted sum
        pass


# TODO: Test and benchmark against other compression methods

### 🏋️ Exercise 3: Frequency-Aware Hash Embedding

Design a hash embedding that dedicates more buckets to high-frequency features and fewer to rare ones.

In [ ]:
class FrequencyAwareHashEmbedding(nn.Module):
    """Hash embedding with frequency-aware bucket allocation.
    
    High-frequency items get dedicated embeddings (no collision).
    Low-frequency items share buckets via hashing.
    
    TODO:
    1. Maintain a "hot" embedding table for top-K most frequent items
    2. Use hash embedding for remaining items
    3. Route items based on frequency threshold
    """
    def __init__(self, num_items, embed_dim, hot_size=1000, cold_buckets=500):
        super().__init__()
        # TODO: Implement frequency-aware hash embedding
        pass
    
    def set_frequency_stats(self, item_counts):
        # TODO: Determine which items are "hot" based on frequency
        pass
    
    def forward(self, ids):
        # TODO: Route to hot or cold embedding
        pass


# TODO: Test with the synthetic data that has power-law frequency distribution

## Summary

In this notebook, we covered practical optimization tricks for ranking models:

| Technique | Memory Savings | Quality Impact | Complexity |
|-----------|---------------|----------------|------------|
| **Hash embedding** | 10-50x | Moderate drop | Low |
| **QR embedding** | 5-20x | Small drop | Low |
| **DHE** | 100x+ | Moderate drop | Medium |
| **Mixed dimensions** | 2-5x | Minimal drop | Low |
| **Mixed precision** | ~2x | Negligible | Low |
| **Gradient compression** | 10-100x comm | Negligible with EF | Medium |

**Key takeaways:**
- Embedding tables dominate ranking model size; compress them first
- Mixed-dimension embeddings provide the best quality-efficiency trade-off
- QR embedding offers consistent quality with significant compression
- Feature importance analysis should guide optimization priorities
- Combining multiple techniques (mixed dim + quantization + compression) yields best results